# LyreVoice — Colab Training

Train the LyreVoice voice cloning GAN on a T4 GPU.

**Prerequisites:** Upload datasets, preprocessed mels, and pretrained checkpoints to Google Drive at `My Drive/lyrevoice/`. See `docs/plans/2026-04-16-colab-migration.md` for the full checklist.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone the repo
!git clone https://github.com/greemwahr/lyrevoice.git /content/lyrevoice
%cd /content/lyrevoice

In [ ]:
# Install dependencies (pip, not uv — Colab doesn't have uv)
!pip install torch torchaudio --quiet
!pip install resemblyzer librosa soundfile pyyaml wandb gradio \
    frechet-audio-distance numpy scipy tqdm matplotlib setuptools --quiet

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Change runtime to T4 GPU.")

In [ ]:
# Verify Drive data is accessible
import os
drive_root = '/content/drive/MyDrive/lyrevoice'
checks = [
    ('Preprocessed VCTK', f'{drive_root}/preprocessed/vctk_metadata.txt'),
    ('Preprocessed LJSpeech', f'{drive_root}/preprocessed/ljspeech_metadata.txt'),
    ('VCTK wavs', f'{drive_root}/datasets/VCTK-Corpus/wav48_silence_trimmed'),
    ('Tacotron2 checkpoint', f'{drive_root}/pretrained/tacotron2_statedict.pt'),
    ('HiFi-GAN checkpoint', f'{drive_root}/pretrained/hifigan_generator.pt'),
    ('HiFi-GAN config', f'{drive_root}/pretrained/hifigan_config.json'),
]
all_ok = True
for name, path in checks:
    exists = os.path.exists(path)
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}] {name}: {path}')
    if not exists:
        all_ok = False
if all_ok:
    print('\nAll data found! Ready to train.')
else:
    print('\nSome data is missing. Upload to Google Drive before training.')

In [ ]:
# Login to WandB
import wandb
wandb.login()

In [ ]:
# Train with Colab config overlay
!python scripts/train.py \
    --config configs/config.yaml \
    --config-override configs/config_colab.yaml

In [ ]:
# Resume training after Colab disconnect
# First, find the latest checkpoint:
!ls -lhrt /content/drive/MyDrive/lyrevoice/checkpoints/ 2>/dev/null || echo 'No checkpoints found yet'

# Uncomment and set the checkpoint path to resume:
# !python scripts/train.py \
#     --config configs/config.yaml \
#     --config-override configs/config_colab.yaml \
#     --resume /content/drive/MyDrive/lyrevoice/checkpoints/lyrevoice_epoch_XXXX.pt